# Zalo AI Challenge - Datamate Inference (bản 0.683) trên Google Colab

Notebook chạy **pipeline chính thức** của repo (`python -m datamate.predict`: vLLM + Qwen3.5-9B + LoRA `sql_rft_v3`), thay cho `LocalSQLPipeline` cũ.

- Clone repo từ GitHub, mount Google Drive, tìm `TEST_DIR` (giữ nguyên như bản cũ).
- Một ô duy nhất: kéo adapter (Git LFS) → cài vLLM → tải model → chạy với cờ của bản nộp, tự chỉnh cho **A100 / L4** (không dùng fp8).
- Kết quả (`submission.jsonl`, `trace.jsonl`, logs) được sao lưu về Drive.

**Cách dùng:** chạy lần lượt từ trên xuống. Ô 5 mặc định `N_TEST = 10` (chạy thử); sau khi ổn thì đặt `N_TEST = None` để chạy toàn bộ.

## 1. Cấu hình & Clone Repo từ GitHub, Mount Google Drive

In [ ]:
import os, sys, subprocess, shutil, getpass
from pathlib import Path
from urllib.parse import urlparse

IN_COLAB = 'google.colab' in sys.modules

# ======================================================================
# 1. THIẾT LẬP REPO GITHUB CỦA BẠN (Thay đổi nếu dùng repo riêng)
# ======================================================================
REPO_URL = "https://github.com/TRUNGNT0/datamate_zaloAI.git"  # <-- URL repo github
BRANCH = "main"
TARGET_DIR = Path("/content/datamate_repo")

if IN_COLAB:
    # Mount Google Drive
    print("▶ Đang kết nối Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # Clone / Pull repo từ GitHub
    print("▶ Đang lấy mã nguồn từ GitHub...")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")  # Token từ Colab Secrets (nếu repo private)
    except Exception:
        pass

    clone_url = REPO_URL
    if token:
        clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

    if (TARGET_DIR / ".git").is_dir():
        print(f"Thư mục {TARGET_DIR} đã tồn tại. Đang cập nhật code mới nhất...")
        subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET_DIR, check=True)
    else:
        print(f"Đang clone {REPO_URL} (branch: {BRANCH})...")
        res = subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            if "Authentication failed" in res.stderr or "could not read Username" in res.stderr:
                token = getpass.getpass("Repo private! Hãy nhập GitHub Personal Access Token (PAT): ")
                clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
                subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], check=True)
            else:
                raise RuntimeError(f"Lỗi clone Git: {res.stderr}")
    ROOT = TARGET_DIR
else:
    ROOT = Path.cwd().resolve()

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Kéo file LFS (nếu adapter được lưu qua Git LFS)
subprocess.run(["git", "lfs", "pull"], cwd=ROOT, check=False)
print(f"✓ Thư mục làm việc: {ROOT}")


## 2. Cài đặt các thư viện phụ thuộc & Kiểm tra GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "⚠️ Không thấy GPU! Runtime > Change runtime type > chọn A100 hoặc L4.")

## 3. Thiết lập Thư mục Lưu Kết Quả & Logs trên Google Drive

In [ ]:
from datetime import datetime

# ĐỊNH NGHĨA ĐƯỜNG DẪN LƯU KẾT QUẢ TRÊN GOOGLE DRIVE
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
if IN_COLAB:
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ZaloAI_Outputs") / f"run_{timestamp}"
else:
    DRIVE_OUTPUT_DIR = ROOT / "outputs" / f"run_{timestamp}"

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)

# Thư mục tạm tốc độ cao trên Colab SSD
LOCAL_RUN_DIR = Path("/content/run_test_tmp" if IN_COLAB else ROOT / "outputs" / "tmp")
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
(LOCAL_RUN_DIR / "logs").mkdir(parents=True, exist_ok=True)

print(f"✓ Thư mục đích lưu kết quả & logs trên Drive: {DRIVE_OUTPUT_DIR}")
print(f"✓ Thư mục chạy tạm thời: {LOCAL_RUN_DIR}")

def sync_to_drive():
    """Đồng bộ ngay lập tức submission.jsonl và toàn bộ logs về Google Drive."""
    if not IN_COLAB:
        return
    for item in LOCAL_RUN_DIR.rglob("*"):
        if item.is_file() and not item.name.endswith(".tmp") and not item.name.endswith(".wal"):
            rel = item.relative_to(LOCAL_RUN_DIR)
            dest = DRIVE_OUTPUT_DIR / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dest)
    print(f"[Sync] Đã sao lưu kết quả & logs mới nhất về Google Drive: {DRIVE_OUTPUT_DIR}", flush=True)

## 4. Xác định Tập Dữ Liệu Test (Public Test hoặc Private Test)

Tìm kiếm thư mục chứa tập test trên repo hoặc trên Google Drive.

In [ ]:
# Tìm kiếm tập test tại các đường dẫn khả dĩ
CANDIDATE_PATHS = [
    ROOT / "dataset" / "public_test",
    ROOT / "data" / "public_test",
    Path("/content/drive/MyDrive/ZaloAI/public_test"),
    Path("/content/drive/MyDrive/dataset/public_test"),
    ROOT / "dataset" / "test",
]

TEST_DIR = None
for p in CANDIDATE_PATHS:
    if p.is_dir() and (p / "questions.jsonl").is_file():
        TEST_DIR = p
        break

# Hoặc bạn có thể chỉ định trực tiếp đường dẫn tập test tại đây:
# TEST_DIR = Path("/content/drive/MyDrive/duong_dan_tap_test")

if TEST_DIR is None:
    print("⚠️ Không tự động tìm thấy questions.jsonl trong các đường dẫn mặc định.")
    print("Hãy đặt đường dẫn tập test vào biến TEST_DIR bên dưới.")
    TEST_DIR = ROOT / "dataset" / "public_test"
else:
    print(f"✓ Đã tìm thấy tập test tại: {TEST_DIR}")

# Kiểm tra số lượng câu hỏi và cấu trúc as_of_date / as_of_day
import json
q_file = TEST_DIR / "questions.jsonl"
if q_file.is_file():
    questions = [json.loads(line) for line in q_file.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    print(f"✓ Tổng số câu hỏi test: {len(questions)}")
    sample = questions[0]
    print(f"✓ Mẫu câu hỏi [0]: ID={sample.get('id')}, Biz={sample.get('business_id')}")
    as_of = sample.get('as_of_date') or sample.get('as_of_day')
    print(f"✓ as_of_date / as_of_day: {as_of}")
    print(f"  Nội dung: {sample.get('question')}")

## 5. Chạy pipeline chính thức (vLLM + Qwen3.5-9B + LoRA)

Ô dưới sẽ: `git lfs pull` adapter (kiểm tra không còn là file con trỏ) → `pip install -e .[llm]` → tải model gốc → chạy `datamate.predict` với cờ của bản nộp (bỏ `--quant`, bỏ KV-cache fp8 cho A100/L4).

> `N_TEST = 10` để chạy thử; `N_TEST = None` để chạy toàn bộ (vài giờ). Log ghi vào `logs/predict.log` và được sync nền lên Drive mỗi 5 phút.

In [ ]:
import os, sys, json, re, subprocess, threading, time, shutil
from pathlib import Path

# ===================== CẤU HÌNH =====================
N_TEST = 10            # None = chạy toàn bộ tập test
CONTEXT_OVERRIDE = None  # vd "8192" nếu bị OOM trên L4; None = tự chọn theo VRAM
# ====================================================

os.chdir(ROOT)
SPLIT = TEST_DIR.name
assert (TEST_DIR / "questions.jsonl").is_file(), f"Không thấy {TEST_DIR}/questions.jsonl — kiểm tra lại TEST_DIR ở ô 4"
OUT = LOCAL_RUN_DIR / SPLIT
OUT.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOCAL_RUN_DIR / "logs" / "predict.log"

def kill_leftovers():
    """Dọn tiến trình vLLM/predict còn sót từ lần Interrupt trước (nếu không, GPU vẫn bị chiếm)."""
    for pat in ("datamate.predict", "VLLM::EngineCore", "vllm"):
        subprocess.run(["pkill", "-9", "-f", pat], capture_output=True)
    time.sleep(3)

def gpu_line():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
            text=True).strip().splitlines()[0]
    except Exception:
        return "?"

def run(cmd, env=None, tee=None, heartbeat=60, **kw):
    """Chạy lệnh, stream output (và ghi log). In 'nhịp tim' mỗi `heartbeat` giây.
    Ctrl-C / Interrupt sẽ kill cả process group con (vLLM) để không chiếm GPU."""
    import signal
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env=env, start_new_session=True, **kw)
    f = open(tee, "a", encoding="utf-8") if tee else None
    t0, last = time.time(), [time.time()]
    stop = threading.Event()
    def _hb():
        while not stop.wait(heartbeat):
            idle = time.time() - last[0]
            files = {x.name: x.stat().st_size for x in OUT.glob("*") if x.is_file()} if OUT.exists() else {}
            print(f"[hb] chạy {(time.time()-t0)/60:.1f} ph | không có log mới {idle:.0f}s | GPU util,mem: {gpu_line()} | OUT: {files}", flush=True)
    if heartbeat:
        threading.Thread(target=_hb, daemon=True).start()
    try:
        for line in p.stdout:
            last[0] = time.time()
            print(line, end="", flush=True)
            if f: f.write(line); f.flush()
    except BaseException:
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception: pass
        raise
    finally:
        stop.set()
        if f: f.close()
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {rc}): {cmd if isinstance(cmd, str) else ' '.join(map(str, cmd))[:300]}")

kill_leftovers()

# ---- 0) GPU ----
try:
    name, mem = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True
    ).strip().splitlines()[0].split(",")
    GPU_NAME, VRAM_GB = name.strip(), float(mem) / 1024
except Exception:
    raise RuntimeError("Không có GPU. Runtime > Change runtime type > A100 (hoặc L4).")
print(f"GPU: {GPU_NAME} ({VRAM_GB:.1f} GB)")
if VRAM_GB < 20:
    raise RuntimeError("VRAM < 20GB: không đủ để nạp Qwen3.5-9B bf16. Dùng A100 hoặc L4.")

# ---- 1) adapter LoRA nằm trong Git LFS: chưa pull thì chỉ là file con trỏ ~134 byte ----
run("apt-get -qq install -y git-lfs > /dev/null && git lfs install --local && git lfs pull", shell=True)
w = ROOT / "adapters/sql_rft_v3/adapter_model.safetensors"
assert w.stat().st_size > 1_000_000, f"{w} vẫn là file con trỏ LFS ({w.stat().st_size} byte): git lfs pull chưa tải được (repo private? kiểm tra GITHUB_TOKEN)"
print(f"✓ Adapter OK ({w.stat().st_size / 1e6:.1f} MB)")

# ---- 2) vLLM (bản repo pin) + model gốc đúng revision ----
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[llm]"])
run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.46.1"])  # chỉ để engine HF dự phòng không crash import
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from datamate.notebook import BASE_MODEL, submitted_flags
from huggingface_hub import snapshot_download
MODEL = ROOT / "models/Qwen3.5-9B"
if not (MODEL / "config.json").exists():
    print("▶ Đang tải Qwen3.5-9B (~19 GB)...")
    snapshot_download(BASE_MODEL[0], revision=BASE_MODEL[1], local_dir=MODEL)
print(f"✓ Model: {MODEL}")

# ---- 3) cờ của bản nộp, chỉnh cho A100/L4 (không fp8) giống Env.inference_flags() ----
flags, engine = submitted_flags(ROOT)
flags = list(flags)
def drop(f, n):
    i = f.index(n) if n in f else -1
    return f if i < 0 else f[:i] + f[i + 2:]
def put(f, n, v):
    return f[:f.index(n) + 1] + [v] + f[f.index(n) + 2:] if n in f else f + [n, v]
def strip_fp8(e):
    try:
        d = json.loads(e); d.pop("kv_cache_dtype", None)
        return json.dumps(d, separators=(",", ":"))
    except Exception:
        return e.replace('"kv_cache_dtype":"fp8",', "")

CONTEXT = CONTEXT_OVERRIDE or ("32768" if VRAM_GB >= 40 else "12288")
flags  = put(drop(flags, "--quant"), "--context", CONTEXT)
engine = strip_fp8(engine)

def tune_for_small_gpu(e):
    """GPU < 40GB (L4 24GB): giảm batch, tắt nhánh vision, bỏ CUDA graph để còn chỗ cho KV cache."""
    try:
        d = json.loads(e)
    except Exception:
        print("⚠️ engine không phải JSON, giữ nguyên:", e[:200]); return e
    d["max_num_seqs"] = min(int(d.get("max_num_seqs", 8)), 8)
    d["max_num_batched_tokens"] = min(int(d.get("max_num_batched_tokens", 4096)), 4096)
    d["gpu_memory_utilization"] = 0.94
    d["enforce_eager"] = True
    d["limit_mm_per_prompt"] = {"image": 0, "video": 0}
    return json.dumps(d, separators=(",", ":"))

if VRAM_GB < 40:
    engine = tune_for_small_gpu(engine)
print(f"✓ context={CONTEXT} | engine={engine[:200]}")

# ---- 4) sync nền log/kết quả lên Drive (mỗi 5 phút) ----
if IN_COLAB and not globals().get("_BG_SYNC_STARTED"):
    def _bg_sync(period=300):
        while True:
            time.sleep(period)
            try:
                for f in LOCAL_RUN_DIR.rglob("*"):
                    if f.is_file() and f.suffix in {".jsonl", ".json", ".log", ".txt", ".csv"}:
                        dest = DRIVE_OUTPUT_DIR / f.relative_to(LOCAL_RUN_DIR)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(f, dest)
            except Exception as e:
                print(f"[bg-sync] lỗi: {e!r}", flush=True)
    threading.Thread(target=_bg_sync, daemon=True).start()
    _BG_SYNC_STARTED = True
    print("✓ Đã bật sync nền (5 phút/lần)")

# ---- 5) chạy pipeline ----
try:
    data_arg = str(TEST_DIR.relative_to(ROOT))   # giống bản gốc: dataset/public_test
except ValueError:
    data_arg = str(TEST_DIR)                     # tập test nằm trên Drive
cmd = [sys.executable, "-m", "datamate.predict",
       "--data", data_arg, "--output", str(OUT),
       "--model", str(MODEL), "--lora", "adapters/sql_rft_v3", "--fewshot", "assets/fewshot_bank.jsonl",
       "--views-from", "configs/public_v9", "--engine", engine, "--save-prompts", *flags]
if N_TEST:
    cmd += ["--limit", str(N_TEST)]
env = {**os.environ, "VLLM_USE_FLASHINFER_SAMPLER": "0", "TOKENIZERS_PARALLELISM": "false",
       "PYTHONHASHSEED": "0", "PYTHONUNBUFFERED": "1"}
print("\n=== BẮT ĐẦU SUY LUẬN ===")
print("FLAGS:", flags)
print("ENGINE:", engine)
print(" ".join(cmd), "\n")
t0 = time.perf_counter()
run(cmd, env=env, tee=LOG_FILE)
print(f"\n✓ Hoàn tất sau {(time.perf_counter() - t0) / 60:.1f} phút")

# ---- 6) kiểm tra định dạng + sao lưu lên Drive ----
from datamate.submission import check
errs = check(OUT / "submission.jsonl", TEST_DIR / "questions.jsonl")
print(errs[:5] or "✓ Định dạng submission OK")
if N_TEST and errs:
    print(f"(N_TEST={N_TEST}: báo thiếu id của phần chưa chạy là bình thường)")
sync_to_drive()

## 6. Kiểm tra & thống kê kết quả (`submission.jsonl` + `trace.jsonl`)

In [ ]:
import json
import pandas as pd

def read_jsonl(p):
    rows = []
    if p.is_file():
        for line in p.read_text(encoding="utf-8", errors="ignore").splitlines():
            if line.strip():
                try: rows.append(json.loads(line))
                except Exception: pass
    return rows

def show(df):
    try: display(df)
    except NameError: print(df.to_string())

sub_rows   = read_jsonl(OUT / "submission.jsonl")
trace_rows = read_jsonl(OUT / "trace.jsonl")

print("=== FILE TRONG THƯ MỤC KẾT QUẢ ===")
for f in sorted(OUT.rglob("*")):
    if f.is_file():
        print(f"✓ {f.relative_to(OUT)} ({f.stat().st_size / 1024:.1f} KB)")

if not sub_rows:
    print("\n⚠️ Chưa có submission.jsonl / file rỗng!")
else:
    df_sub = pd.DataFrame(sub_rows)
    print(f"\n✓ Tổng số dự đoán: {len(df_sub)}")
    if "type" in df_sub:
        print("\nPhân bố loại đáp án:")
        print(df_sub["type"].value_counts())

if trace_rows:
    df_tr = pd.DataFrame(trace_rows)
    cols = [c for c in ("id", "business_id", "question", "final", "vote", "conf") if c in df_tr.columns]
    print(f"\n✓ Trace: {len(df_tr)} dòng | cột: {list(df_tr.columns)}")
    show(df_tr[cols].head(10) if cols else df_tr.head(10))
else:
    print("\n(Chưa có trace.jsonl)")

In [ ]:
# Tải submission.jsonl về máy (tùy chọn)
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT / "submission.jsonl"))